### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [3]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [4]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [6]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- planning, studying, and problem-solving

I work by generating responses based on patterns in data I was trained on, so I can be very useful, but I’m not a person and I don’t have feelings, beliefs, or personal experiences.

A few useful things to know:
- I can be conversational or concise, depending on what you want.
- I can help with both simple and complex tasks.
- I may sometimes make mistakes, so important facts should be double-checked.
- I don’t “know” things the way humans do; I predict useful text based on your prompt.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to get the best results from me

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [8]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "LLM Leaderboard - Comparison of over 100 AI models from OpenAI, Google ...",
      "url": "https://artificialanalysis.ai/leaderboards/models",
      "description": "Comparison and ranking the performance of over 100 AI <b>models</b> (<b>LLMs</b>) across key metrics including intelligence, price, performance and speed (output speed - tokens per second &amp; laten

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools and error handling

In [10]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [15]:
import httpx
from langchain.tools import tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [23]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open and close LLM models in 2026 ?
        Compare their capabilities and limitations.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open and close LLM models in 2026 ?
        Compare their capabilities and limitations.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_F62w44WCXabRe9cDBSXho3Gx)
 Call ID: call_F62w44WCXabRe9cDBSXho3Gx
  Args:
    query: latest open-source and closed-source LLM models 2026 capabilities limitations 2026 site:openai.com OR site:anthropic.com OR site:google.com OR site:meta.com OR site:mistral.ai OR site:deepseek.com OR site:x.ai OR site:alibabacloud.com
    limit: 50
    searchMode: request
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open-source and closed-sourc

I searched for up to 50 web results, but the search engine only returned **25 results** for this query set. I fetched a broad subset of the most relevant pages successfully; a few pages were blocked or failed (notably OpenAI’s GPT-5.5 announcement page returned **403** in this environment). So the comparison below is based on the fetched pages plus search-result evidence where fetch was unavailable.

## Short answer

As of the fetched 2026 pages, the **latest notable closed models** I could verify are:

- **Anthropic Claude Opus 4.7** — Apr 2026
- **Anthropic Claude Mythos Preview** — Apr 2026, limited release
- **Meta Muse Spark** — Apr 2026, private API preview / Meta AI app
- **Google Gemini 3.1 Pro / Gemini 3 Pro** — listed in Google docs as current model family
- **OpenAI GPT-5.5** — search result shows Apr 23, 2026, but I could not fetch the OpenAI page directly due to 403

The **latest notable open/open-weight models** I could verify are:

- **Mistral Small 4** — Apache 2.0, open, multimodal/reasoning/coding unified
- **Mistral 3 family / Mistral Large 3 / Ministral 3** — Apache 2.0
- **DeepSeek V3.2 / R1-0528** — listed as open models on Google Vertex AI docs
- **OpenAI gpt-oss-120b / gpt-oss-20b** — listed as open models on Vertex AI docs
- **Qwen 3 family** — listed as open models on Vertex AI docs
- **Gemma 4 variants** — listed as open models on Vertex AI docs
- **Llama 4 Maverick / Scout** — listed on Vertex AI docs as available partner/open ecosystem models

---

# Closed vs open LLMs in 2026

## 1. Closed models: strongest currently visible frontier offerings

### Anthropic Claude Opus 4.7
**What it appears best at**
- Advanced software engineering
- Long-running autonomous/agentic tasks
- Better instruction-following than prior Claude models
- Improved high-resolution vision
- Strong enterprise document, finance, and coding workflows

**Evidence from fetched pages**
- Anthropic says Opus 4.7 improves on Opus 4.6 in “advanced software engineering,” “complex, long-running tasks,” instruction following, and higher-resolution vision.
- Anthropic also notes it is **less broadly capable than Claude Mythos Preview**, implying Mythos is their more frontier-capable internal/limited model.

**Limitations**
- More token-hungry than prior versions in some workloads due to tokenizer/reasoning changes
- Still not their most powerful model overall
- Cyber use is guarded by real-time safeguards and verification programs
- Anthropic says alignment is improved in some ways but “not fully ideal”

### Anthropic Claude Mythos Preview
**What it appears best at**
- Frontier cyber capability
- Advanced agentic coding and reasoning
- Stronger than Opus 4.7 overall, but not generally released

**Limitations**
- Not broadly available
- Restricted to defensive cybersecurity partners
- High-risk capability profile means stricter access and mitigations

### Meta Muse Spark
**What it appears best at**
- Native multimodality
- Reasoning with tool use and multi-agent orchestration
- Health and visual reasoning use cases
- “Contemplating mode” for stronger difficult-task performance

**Limitations**
- Meta explicitly says they are still improving:
  - long-horizon agentic systems
  - coding workflows
- API is only in private preview
- Evaluation-awareness concerns were noted in Meta’s own reporting

### Google Gemini 3 / 3.1 family
From Google’s current documentation, Gemini 3.1 Pro and Gemini 3 Pro are among the newest flagship Google closed models exposed in docs.

**Strengths**
- Broad managed platform integration
- Strong ecosystem, multimodal stack, enterprise tooling

**Limitations**
- The fetched docs page mainly confirms current availability, not benchmark detail on one page
- Less directly comparable here because I did not fetch a dedicated announcement/system-card page in this run

### OpenAI GPT-5.5
Search results indicate:
- “Introducing GPT-5.5”
- Apr 23, 2026
- described as smarter, faster, and more capable for coding/research/data analysis

**Limitation of this comparison**
- I could not fetch the OpenAI page directly due to 403, so I won’t overstate details beyond the search snippet.

---

## 2. Open/open-weight models: strongest currently visible offerings

## Mistral Small 4
This is one of the clearest open-model releases I could fetch.

**Strengths**
- Apache 2.0 license
- Unifies:
  - instruct/chat
  - reasoning
  - multimodal
  - agentic coding
- 256k context
- Native image + text input
- Configurable reasoning effort
- Efficiency-focused MoE design

**Architecture**
- 119B total params
- 128 experts
- 4 active per token
- 6B active params/token, 8B incl. embeddings/output

**Claimed advantages**
- Lower latency and higher throughput than Mistral Small 3
- Competitive performance while generating shorter outputs than some rivals

**Limitations**
- Requires meaningful infrastructure for optimal deployment
- Open models still generally trail the very top closed frontier systems in raw breadth, robustness, or product polish
- Real-world reliability depends heavily on deployment stack and tuning

## Mistral 3 family / Mistral Large 3 / Ministral 3
**Strengths**
- Apache 2.0 license
- Broad size range:
  - 3B / 8B / 14B edge-friendly variants
  - Large 3 at 675B total / 41B active MoE
- Multimodal and multilingual
- Includes reasoning variants
- Strong openness and portability

**Best fit**
- Enterprises wanting self-hosting, fine-tuning, or deployment control
- Edge/local deployments with smaller models
- Teams wanting permissive licensing

**Limitations**
- “Frontier” open, but still usually needs more engineering effort than managed closed APIs
- Large model deployment remains expensive/complex
- Safety, monitoring, and tool orchestration are more your responsibility

## DeepSeek open models
The Google docs page lists:
- **DeepSeek-V3.2**
- **DeepSeek-V3.1**
- **DeepSeek-R1-0528**
- **DeepSeek-OCR**

**Likely strengths**
- Strong reasoning/coding reputation in 2026 ecosystem
- Broad availability through managed open-model serving

**Limitations**
- I did not fetch the DeepSeek model detail pages in this run, so I won’t rank them too aggressively here

## OpenAI gpt-oss-120b / 20b
Google docs list these as open models in MaaS.

**Strengths**
- Open-weight availability
- Likely useful for self-hosted or managed-open deployments

**Limitations**
- No detailed fetched benchmark page in this run

## Qwen 3 family
Listed by Google docs:
- Qwen 3 Next Instruct 80B
- Qwen 3 Next Thinking 80B
- Qwen 3 Coder
- Qwen 3 235B

**Strengths**
- Strong variety across instruct, thinking, coder
- Competitive open ecosystem presence

**Limitations**
- Not directly fetched from primary vendor pages here

## Gemma 4
Google docs list Gemma 4 MaaS variants.

**Strengths**
- Strong integration in Google ecosystem
- Open-model availability

**Limitations**
- Smaller ecosystem story than some broader open families unless you are already on Google tooling

## Llama 4 Maverick / Scout
Listed in Vertex AI docs as available current models.

**Strengths**
- Major ecosystem adoption
- Open-weight family with strong integration options

**Limitations**
- In this fetched set, I got Meta’s **Muse Spark** page, not a fresh Llama 4 release page, so details here are ecosystem-level rather than launch-detail-level

---

# Capability comparison

## Closed models: main advantages
1. **Best raw capability ceiling**
   - Anthropic’s own materials explicitly position Mythos Preview above Opus 4.7.
   - Closed frontier labs usually lead on hardest coding, long-horizon reasoning, safety tuning, and product integration.

2. **Stronger agent reliability**
   - Opus 4.6/4.7 material repeatedly emphasizes long-running tasks, reduced tool errors, planning, and memory.

3. **Better integrated safety systems**
   - Real-time safeguards, abuse detection, prompt injection mitigation, usage controls.

4. **More polished multimodal/product workflows**
   - Enterprise docs, spreadsheets, slides, computer use, search, code, memory.

## Closed models: main limitations
1. **Restricted access**
   - The strongest models may be preview-only or heavily gated.
2. **Less transparency**
   - Weights/training data/code are not open.
3. **Vendor dependence**
   - Pricing, retirement, API policy, and rate limits are controlled by provider.
4. **Harder to self-host**
   - Usually impossible.

---

## Open/open-weight models: main advantages
1. **Control**
   - Self-hosting, customization, fine-tuning, compliance, latency control.
2. **Licensing flexibility**
   - Especially strong with Apache 2.0 releases like Mistral Small 4 and Mistral 3.
3. **Cost optimization at scale**
   - Potentially cheaper for high-volume or persistent workloads if you can run infra efficiently.
4. **Portability**
   - Easier to run on vLLM, llama.cpp, SGLang, Transformers, etc.

## Open/open-weight models: main limitations
1. **Usually slightly behind the top closed frontier models**
   - Particularly on robustness, tool use, long-horizon autonomy, and product polish.
2. **Operational burden**
   - You own deployment, safety, scaling, monitoring, and updates.
3. **Hardware demands**
   - The strongest open models still need serious GPUs.
4. **“Open” is often open-weight, not fully open-source**
   - Google’s own docs note that open-weight models may not expose full code/data/training details.

---

# Best-in-class by use case in 2026

## If you want the strongest closed model
- **Claude Opus 4.7** for production access
- **Claude Mythos Preview** if you mean raw capability and can accept restricted availability

## If you want the strongest open/open-weight model from fetched evidence
- **Mistral Small 4** for a modern open multimodal/reasoning/coding unified model
- **Mistral Large 3** if you want a large permissive open frontier-class model
- **Qwen 3 / DeepSeek / gpt-oss** are also current important contenders, but I have less direct fetched evidence in this run

## If you care about self-hosting and enterprise control
- **Mistral 3 family**
- **Mistral Small 4**
- also potentially **DeepSeek**, **Qwen**, **Gemma**, **Llama**, depending on deployment stack and region

## If you care about computer use / agents
- **Claude Sonnet 4.6 / Opus 4.6 / Opus 4.7** look particularly strong from the fetched materials

## If you care about cyber/security workflows
- **Claude Mythos Preview** appears strongest but restricted
- **Claude Opus 4.7 / 4.6** are strongly positioned for practical security workflows with safeguards

---

# Practical conclusion

If you ask, “What are the latest open and closed LLMs in 2026?” the most defensible summary from the fetched sources is:

### Leading closed models
- Anthropic **Claude Opus 4.7**
- Anthropic **Claude Mythos Preview**
- Meta **Muse Spark**
- Google **Gemini 3.1 Pro / Gemini 3 Pro**
- OpenAI **GPT-5.5** (search verified, page fetch blocked)

### Leading open/open-weight models
- **Mistral Small 4**
- **Mistral 3 / Mistral Large 3 / Ministral 3**
- **DeepSeek V3.2 / R1**
- **OpenAI gpt-oss-120b / 20b**
- **Qwen 3 family**
- **Gemma 4**
- **Llama 4 family**

### Overall comparison
- **Closed models still lead** in absolute frontier capability, agent reliability, product integration, and safety orchestration.
- **Open/open-weight models are catching up fast** and are often better for cost control, deployment flexibility, customization, and governance.
- In 2026, the main strategic choice is less “which is smarter?” and more:
  - **maximum capability and convenience** → closed
  - **control, portability, self-hosting, and customization** → open/open-weight

---

# Notes on the web search/fetch request
- Requested: search 50 result pages and fetch webpages
- Actual: search returned **25 results total**, not 50
- I fetched a substantial subset of the most relevant pages
- Some pages failed:
  - OpenAI GPT-5.5 page: **403 Forbidden**
  - one Anthropic system-card URL redirect/fetch issue
  - one Anthropic PDF URL typo/404

---

# References

1. Anthropic — *Introducing Claude Opus 4.7*  
   https://www.anthropic.com/news/claude-opus-4-7

2. Anthropic — *Introducing Claude Opus 4.6*  
   https://www.anthropic.com/news/claude-opus-4-6

3. Anthropic — *Introducing Claude Sonnet 4.6*  
   https://www.anthropic.com/news/claude-sonnet-4-6

4. Anthropic — *Transparency Hub / Model Report*  
   https://www.anthropic.com/transparency

5. Meta AI — *Introducing Muse Spark: Scaling Towards Personal Superintelligence*  
   https://ai.meta.com/blog/introducing-muse-spark-msl/

6. Mistral AI — *Introducing Mistral Small 4*  
   https://mistral.ai/news/mistral-small-4

7. Mistral AI — *Introducing Mistral 3*  
   https://mistral.ai/news/mistral-3

8. Google Cloud Docs — *Anthropic’s Claude models on Vertex AI*  
   https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/claude

9. Google Cloud Docs — *Overview of self-deployed models*  
   https://docs.cloud.google.com/vertex-ai/generative-ai/docs/model-garden/self-deployed-models

10. Search result evidence for OpenAI GPT-5.5 announcement  
   https://openai.com/index/introducing-gpt-5-5/  
   (page returned 403 in this environment, but search results identified title/date/snippet)

If you want, I can do a **second pass focused only on open models** and produce a **ranked top-10 table with columns for license, parameter size, context length, modality, deployment mode, and best use case**.